# 七瀬つむぎさんの出演歴とサンプルボイスの取得

ホーリーピーク公式ページの見出しごとに出演歴を分け、`<audio>` のURLと共に辞書化します。

In [ ]:
from pprint import pprint
from urllib.parse import urljoin

import requests
from bs4 import BeautifulSoup, Tag

URL = ("https://holypeak.com/talent/voice-actor-women/"
       "%e4%b8%83%e7%80%ac-%e3%81%a4%e3%82%80%e3%81%8e/")
HEADERS = {"User-Agent": "VoiceActorStudyScraper/1.0 (educational use)"}
EXCLUDED_HEADINGS = {"プロフィール", "サンプルボイス"}


def siblings_until_next_h2(heading):
    """現在のh2から次のh2までの要素を返す。"""
    for sibling in heading.next_siblings:
        if isinstance(sibling, Tag) and sibling.name == "h2":
            break
        if isinstance(sibling, Tag):
            yield sibling


def extract_entries(element):
    """p内のbr、またはul/ol内のliを1件ずつに分ける。"""
    if element.name == "p":
        return [line.strip() for line in
                element.get_text("\n", strip=True).splitlines() if line.strip()]
    if element.name in {"ul", "ol"}:
        return [li.get_text(" ", strip=True)
                for li in element.find_all("li", recursive=False)
                if li.get_text(" ", strip=True)]

    entries = []  # divなどで囲まれた場合にも対応
    for child in element.find_all(["p", "ul", "ol"]):
        entries.extend(extract_entries(child))
    return entries


def scrape_voice_actor(url):
    result = {"appearances": {}, "voice_samples": []}

    try:
        # 1回のみリクエストし、接続と読み込みにタイムアウトを設ける
        response = requests.get(url, headers=HEADERS, timeout=(5, 20))
        response.raise_for_status()
    except requests.exceptions.Timeout as exc:
        print(f"タイムアウトしました: {exc}")
        return result
    except requests.exceptions.RequestException as exc:
        print(f"ページの取得に失敗しました: {exc}")
        return result

    try:
        soup = BeautifulSoup(response.content, "html.parser")
        content = soup.select_one("div.container__contents.wysiwyg")
        if content is None:
            raise ValueError("本文領域が見つかりません")

        for heading in content.find_all("h2", recursive=False):
            category = heading.get_text(" ", strip=True)
            blocks = list(siblings_until_next_h2(heading))

            if category == "サンプルボイス":
                for block in blocks:
                    for media in block.select("audio[src], audio source[src]"):
                        audio_url = urljoin(response.url, media.get("src"))
                        if audio_url not in result["voice_samples"]:
                            result["voice_samples"].append(audio_url)
                continue

            if category not in EXCLUDED_HEADINGS:
                entries = []
                for block in blocks:
                    entries.extend(extract_entries(block))
                if entries:
                    result["appearances"][category] = entries

    except (ValueError, AttributeError, TypeError) as exc:
        print(f"HTMLの解析に失敗しました: {exc}")

    return result


data = scrape_voice_actor(URL)
pprint(data, sort_dicts=False)

{'appearances': {'アニメ': ['「宇宙人ムームー」園児'],
                 'ライブ': ['「学園アイドルマスター LIVE TOUR –標- 福井公演」（2026.8.20-8.21 '
                         'フェニックス・プラザ エルピス 大ホール）',
                         '「学園アイドルマスター\xa0The 2nd Period Hatsuboshi IDOL '
                         'FESTIVAL」（2026.6.6-6.7 横浜アリーナ）',
                         '「学園アイドルマスター The 2nd Period\xa0'
                         'H.I.F選抜試験(セレクション)」（2026.5.16 幕張メッセ・イベントホール）',
                         '「学園アイドルマスター 初星音楽祭」（2026.2.28-3.1 京王アリーナTOKYO）',
                         '「THE IDOLM@STER 20th ANNIVERSARY NEW YEAR MEETING & '
                         'DJ PARTY」（2026.1.3 Zepp Osaka Bayside）',
                         '「学園アイドルマスター クラス対抗初星大運動会」（2025.9.20-9.21 国立代々木競技場 '
                         '第一体育館）',
                         '「学園アイドルマスター\xa0The 1st Period Harmony '
                         'Star」（2025.5.31-6.1 立川ステージガーデン）',
                         '「学園アイドルマスター\xa0The 1st Period Spotlight '
                         'Star」（2025.5.24-5.

## JSONファイルへの保存

スクレイピングで取得した辞書型データを、日本語のまま読みやすいJSONファイルとして保存します。

- `ensure_ascii=False`: これを指定しないと、日本語が `\u5b87\u5b99...` のようなUnicodeエスケープシーケンスに変換されてしまい、ファイルを開いたときに人間が読みにくくなります。
- `indent=4`: 4文字分のインデントと改行が入り、データの階層構造が見やすい形で保存されます。

In [ ]:
import json

# スクレイピングセルで作成した辞書を保存する
scraped_data = data

with open("nanase_tsumugi_data.json", "w", encoding="utf-8") as f:
    json.dump(scraped_data, f, ensure_ascii=False, indent=4)

print("JSONファイルへの保存が完了しました！")

JSONファイルへの保存が完了しました！
